# CRISPRtracrRNA Prediction

![CRISPRtracrRNA Prediction](https://proto-bio.github.io/proto-assets/images/tool/crispr_tracr_rna/hero.png)

This notebook demonstrates `run_crispr_tracr_rna`, which scans nucleotide sequences for tracrRNA candidates using the CRISPRtracrRNA tool from the Backofen Lab. tracrRNA is required by type II CRISPR-Cas9 systems to form the guide RNA complex; identifying it is a critical step when characterizing novel Cas9 orthologs or validating candidate loci in a CRISPR engineering pipeline.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("crispr_tracr_rna")
display_overview("crispr_tracr_rna")
display_docs_section("crispr_tracr_rna", "Background")

# CRISPRtracrRNA

[CRISPRtracrRNA](https://github.com/BackofenLab/CRISPRtracrRNA) is a multi-evidence pipeline from the Backofen Lab that predicts [tracrRNA](https://en.wikipedia.org/wiki/Trans-activating_crRNA) sequences in nucleotide [CRISPR](https://en.wikipedia.org/wiki/CRISPR) loci. It combines covariance-model search, CRISPR array detection, cas-effector cassette detection, anti-repeat similarity, RNA-RNA interaction prediction, and terminator detection into a weighted multi-evidence ranking score.

## Background

**What does this tool measure/predict?**
Per-locus tracrRNA candidates with full evidence: candidate position and sequence, CRISPR array context, anti-repeat similarity to the array's repeat consensus, predicted RNA-RNA interaction with the CRISPR repeat, transcription terminator location, distance to the nearest cas-effector cassette, and a single weighted multi-evidence ranking score.

**Why is this important?**
For a [CRISPR-Cas9](https://en.wikipedia.org/wiki/CRISPR#Class_2) system to function, three components must be present: the [Cas9](https://en.wikipedia.org/wiki/Cas9) effector, the [crRNA](https://en.wikipedia.org/wiki/CRISPR#crRNA) (from the array), and the tracrRNA. Detecting tracrRNA is essential for confirming a complete Type II locus, designing [single-guide RNAs](https://en.wikipedia.org/wiki/Guide_RNA) by fusing crRNA + tracrRNA, validating computationally generated CRISPR systems, and discovering novel Cas9/Cas12 systems in metagenomic data.

**Scientific foundation:**
Mitrofanov et al. (2022) — multi-evidence integration is more sensitive than covariance-model search alone, especially for divergent tracrRNA families and novel CRISPR-Cas systems.

## Available tools

In [2]:
display_available_tools("crispr_tracr_rna")

- **`run_crispr_tracr_rna()`** — Predict tracrRNA sequences from nucleotide CRISPR loci

### `run_crispr_tracr_rna`

`run_crispr_tracr_rna` predicts tracrRNA sequences from nucleotide sequences that contain CRISPR loci. The tool evaluates anti-repeat complementarity and uses IntaRNA to compute RNA-RNA interaction energies, returning predicted tracrRNA coordinates and interaction energy scores for each input sequence. In practice the input should be a confirmed CRISPR-containing genomic region, typically identified by a tool like MinCED in an earlier analysis step.

In [3]:
from proto_tools import (
    CrisprTracrRNAInput,
    CrisprTracrRNAConfig,
    run_crispr_tracr_rna,
)

In [4]:
# Display input docs
display_api_reference("crispr_tracr_rna", "input", "run_crispr_tracr_rna")

# Provide a genomic sequence containing a CRISPR locus
# In practice, use a real CRISPR-containing genomic region
inputs = CrisprTracrRNAInput(
    sequences=["ATGCGTAAACGATTGCAGT" * 500],
)

**Input** — `CrisprTracrRNAInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `list[str]` | required | Nucleotide sequence(s) to predict tracrRNA from |

In [5]:
# Display config docs
display_api_reference("crispr_tracr_rna", "config", "run_crispr_tracr_rna")

# Type II model for Cas9-associated tracrRNA prediction | see docs above for all fields
config = CrisprTracrRNAConfig(model_type="II")

**Config** — `CrisprTracrRNAConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cpu'` | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| `model_type` | `Literal['II', 'all']` | `'II'` | CRISPR model: 'II' (type II only, fast, default) or 'all' (type II + type V cluster models) |
| `run_type` | `Literal['complete_run', 'model_run']` | `'complete_run'` | Pipeline mode: 'complete_run' (full ranking, default) or 'model_run' (cmsearch scan only, fast) |
| `num_workers` | `int \| None` | `None` | Parallel workers across input sequences. None or 0 defaults to 1; capped at len(sequences). |
| `anti_repeat_similarity_threshold` | `float` | `0.7` | Anti-repeat ↔ repeat similarity floor (0-1). Lower for divergent CRISPR families |
| `anti_repeat_coverage_threshold` | `float` | `0.6` | Anti-repeat alignment coverage floor (0-1). Lower for partial anti-repeats |
| `weight_crispr_array_score` | `float` | `0.5` | Multi-evidence ranking weight for CRISPRidentify array-detection confidence. |
| `weight_anti_repeat_sim` | `float` | `0.5` | Multi-evidence ranking weight for anti-repeat sequence similarity. |
| `weight_anti_repeat_coverage` | `float` | `0.5` | Multi-evidence ranking weight for anti-repeat alignment coverage. |
| `weight_anti_sim_coverage` | `float` | `0.5` | Multi-evidence ranking weight for the similarity x coverage product. |
| `weight_interaction_score` | `float` | `0.6` | Multi-evidence ranking weight for the IntaRNA RNA-RNA interaction energy. |
| `weight_model_hit_score` | `float` | `0.9` | Multi-evidence ranking weight for the covariance-model tail hit score. |
| `weight_terminator_hit_score` | `float` | `0.9` | Multi-evidence ranking weight for erpin terminator presence/score. |
| `weight_consistency_orientation` | `float` | `0.1` | Multi-evidence ranking weight for repeat / anti-repeat orientation consistency. |
| `weight_consistency_anti_repeat_tail` | `float` | `0.1` | Multi-evidence ranking weight for anti-repeat ↔ tail positional consistency. |
| `weight_consistency_tail_terminator` | `float` | `0.1` | Multi-evidence ranking weight for tail ↔ terminator positional consistency. |
| `perform_type_v_anti_repeat_analysis` | `bool` | `False` | Search Type V (Cas12) anti-repeat locations. Niche; off by default |

In [6]:
# Run the tracrRNA prediction
result = run_crispr_tracr_rna(inputs, config)

Running run_crispr_tracr_rna [00:00]

In [7]:
# Display output docs
display_api_reference("crispr_tracr_rna", "output", "run_crispr_tracr_rna")

# One CrisprTracrRNASequenceResult per input sequence; each carries every candidate hit
# upstream produced for that accession, sorted by score descending.
for seq_result in result.results:
    top = seq_result.top_candidate
    if top is not None and top.has_tracr:
        print(f"{seq_result.sequence_id}: tracrRNA at {top.anti_repeat_start}-{top.anti_repeat_end} "
              f"(score={top.score}, {len(seq_result.candidates)} candidates)")
        print(f"  Interaction energy: {top.interaction_energy} kcal/mol")
    else:
        print(f"{seq_result.sequence_id}: no tracrRNA detected")

**Output** — `CrisprTracrRNAOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `results` | `list[proto_tools.tools.gene_annotation.crispr_tracr_rna.crispr_tracr_rna.CrisprTracrRNASequenceResult]` | `[]` | Per-input sequence results, each with all candidate hits top-ranked first. |

locus_1: no tracrRNA detected


## Export Results

TracrRNA predictions can be exported to CSV or JSON. Both formats flatten to one row per candidate hit — sequences with multiple candidates produce multiple rows (sorted by `score` descending), and sequences with no candidates produce a single `sequence_id`-only row.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="crispr_tracr_rna_results", export_path=output_dir, file_format="json")
print(f"tracrRNA predictions exported to {output_dir}")

tracrRNA predictions exported to example_output
